In [ ]:
# @title 🏢 대한민국 데이터센터 입지 통합 분석 (Final Clean Version)
!pip install -q geemap osmnx ipywidgets ipyleaflet

import ee, geemap, osmnx as ox
import ipywidgets as widgets
from ipywidgets import Layout, HBox, VBox
from ipyleaflet import LayerGroup, Marker, AwesomeIcon

# [1] 인증 및 초기화
try: ee.Initialize(project='YOUR_EARTH_ENGINE_PROJECT_ID')
except: ee.Authenticate(); ee.Initialize(project='YOUR_EARTH_ENGINE_PROJECT_ID')

# ==========================================
# 1. 📍 데이터 정의 (신규 10곳 + 기존 6곳)
# ==========================================
sites = {
    'New': [
        {'loc':'충남 부여(1위)', 'pos':[36.25,126.98]}, {'loc':'경기 파주(2위)', 'pos':[37.97,126.98]},
        {'loc':'경기 용인(3위)', 'pos':[37.11,127.16]}, {'loc':'강원 고성(4위)', 'pos':[38.04,128.05]},
        {'loc':'인천 강화(5위)', 'pos':[37.68,126.44]}, {'loc':'충남 서천(6위)', 'pos':[36.18,126.80]},
        {'loc':'경기 안성(7위)', 'pos':[37.04,127.43]}, {'loc':'강원 강릉(8위)', 'pos':[37.68,128.86]},
        {'loc':'부산 기장(9위)', 'pos':[35.38,129.22]}, {'loc':'전남 구례(10위)', 'pos':[35.16,127.43]}
    ],
    'Old': [
        {'loc':'네이버 춘천', 'pos':[37.91,127.77]}, {'loc':'카카오 판교', 'pos':[37.40,127.11]},
        {'loc':'삼성SDS 상암', 'pos':[37.58,126.89]}, {'loc':'네이버 세종', 'pos':[36.56,127.26]},
        {'loc':'LG CNS 부산', 'pos':[35.14,128.90]}, {'loc':'KT 목동', 'pos':[37.53,126.88]}
    ]
}

# 16개 지역 반경 20km 영역 결합 (통합 ROI)
all_points = [x['pos'] for x in sites['New'] + sites['Old']]
roi_all = ee.Geometry.MultiPolygon([ee.Geometry.Point([p[1], p[0]]).buffer(20000) for p in all_points])

# ==========================================
# 2. 🛰️ 위성 분석 (바다 투명화 & GSI)
# ==========================================
# 데이터 로드
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG").filterDate('2023-01-01','2023-12-31').select('avg_rad').mean()
lst = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterDate('2023-06-01','2023-09-30').select('ST_B10').mean().multiply(0.003418).add(149).subtract(273.15)
wc = ee.ImageCollection("ESA/WorldCover/v200").first().select('Map')
plants = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([129.35,37.24])), ee.Feature(ee.Geometry.Point([126.52,36.40])), ee.Feature(ee.Geometry.Point([129.29,35.32]))])

# GSI 모델링
norm = lambda img, mn, mx: img.clamp(mn, mx).subtract(mn).divide(mx - mn)
gsi = (ee.Image(1).subtract(norm(plants.distance(200000).divide(1000), 80, 150)).multiply(0.25)) \
      .add(ee.Image(1).subtract(norm(viirs, 0, 50)).multiply(0.20)) \
      .add(ee.Image(1).subtract(norm(lst, 20, 35)).multiply(0.15)) \
      .add(wc.remap([10,20,30,40,50,60,80], [0.3,0.7,1.0,0.6,0.0,1.0,0.0], 0.0).multiply(0.20)) \
      .add(ee.Image(1).subtract(norm(ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003")), 0, 15)).multiply(0.20))

# 🌊 바다 마스킹 및 도구 정의
mask = wc.neq(80)
tools = {
    '📊 GSI 적합도': (gsi.updateMask(mask), {'min':0.5, 'max':0.9, 'palette':['red','orange','yellow','green','blue']}),
    '💡 야간조도': (viirs.updateMask(mask), {'min':0, 'max':10, 'palette':['black','purple','cyan','white']}),
    '🌡️ 지표온도': (lst.updateMask(mask), {'min':20, 'max':35, 'palette':['blue','green','yellow','red']}),
    '🌲 토지피복': (wc.updateMask(mask), {'min':10, 'max':100, 'palette':['006400','ffbb22','fa0000','0064c8']}),
    '🏆 최적 후보지': (gsi.gt(0.78).updateMask(mask).selfMask(), {'palette':['00FF00']})
}

# ==========================================
# 3. 🗺️ 지도 시각화 및 제어 UI
# ==========================================
m = geemap.Map(center=[36.5, 127.5], zoom=7)

# [A] 분석 레이어 추가 (초기엔 모두 숨김)
for name, (img, vis) in tools.items():
    m.addLayer(img.clip(roi_all), vis, name, False)

# [B] 마커 추가 (항상 표시: 신규=초록별, 기존=검정건물)
g_markers = LayerGroup()
for s in sites['New']:
    g_markers.add_layer(Marker(location=s['pos'], icon=AwesomeIcon(name='star', marker_color='green', icon_color='white'), title=s['loc'], draggable=False))
for s in sites['Old']:
    g_markers.add_layer(Marker(location=s['pos'], icon=AwesomeIcon(name='building', marker_color='black', icon_color='white'), title=s['loc'], draggable=False))
m.add_layer(g_markers)

# [C] 컨트롤러 (드롭다운 + 버튼)
dropdown = widgets.Dropdown(options=tools.keys(), value='📊 GSI 적합도', description='🛠️ 도구:', layout=Layout(width='250px'))
btn_osm = widgets.Button(description='⚡ 송전망 보기', icon='bolt', button_style='warning')

# [D] 업데이트 로직
def update_layers(change=None):
    selected = dropdown.value
    for layer in m.layers:
        if layer.name in tools.keys(): # 도구 레이어만 제어
            layer.visible = (layer.name == selected)

def load_osm(b):
    try:
        print("⏳ 송전선 로딩 중... (화면 중심 반경 20km)")
        lines = ox.features_from_point((m.center[0], m.center[1]), tags={'power':['line','cable']}, dist=20000)
        m.add_gdf(lines[lines.geometry.type.isin(['LineString','MultiLineString'])], layer_name='⚡ 송전선로', style={'color':'red', 'weight':3})
    except: print("⚠️ 송전선 데이터 없음")

dropdown.observe(update_layers, 'value')
btn_osm.on_click(load_osm)
update_layers() # 초기 실행

print("✅ 준비 완료. 드롭다운 메뉴로 분석 도구를 변경하세요.")
display(HBox([dropdown, btn_osm]))
display(m)

In [ ]:
# @title 🏢 대한민국 데이터센터 입지 통합 분석 (Perfect Simple)
!pip install -q geemap osmnx ipywidgets ipyleaflet

import ee, geemap, osmnx as ox
import ipywidgets as widgets
from ipywidgets import Layout, HBox, VBox
from ipyleaflet import LayerGroup, Marker, AwesomeIcon

# 인증 및 초기화
try: ee.Initialize(project='YOUR_EARTH_ENGINE_PROJECT_ID')
except: ee.Authenticate(); ee.Initialize(project='YOUR_EARTH_ENGINE_PROJECT_ID')

# ==========================================
# 1. 📍 데이터 정의 (신규 10곳, 기존 6곳)
# ==========================================
sites = {
    'New': [
        {'loc':'충남 부여(1위)', 'pos':[36.25,126.98]}, {'loc':'경기 파주(2위)', 'pos':[37.97,126.98]},
        {'loc':'경기 용인(3위)', 'pos':[37.11,127.16]}, {'loc':'강원 고성(4위)', 'pos':[38.04,128.05]},
        {'loc':'인천 강화(5위)', 'pos':[37.68,126.44]}, {'loc':'충남 서천(6위)', 'pos':[36.18,126.80]},
        {'loc':'경기 안성(7위)', 'pos':[37.04,127.43]}, {'loc':'강원 강릉(8위)', 'pos':[37.68,128.86]},
        {'loc':'부산 기장(9위)', 'pos':[35.38,129.22]}, {'loc':'전남 구례(10위)', 'pos':[35.16,127.43]}
    ],
    'Old': [
        {'loc':'네이버 춘천', 'pos':[37.91,127.77]}, {'loc':'카카오 판교', 'pos':[37.40,127.11]},
        {'loc':'삼성SDS 상암', 'pos':[37.58,126.89]}, {'loc':'네이버 세종', 'pos':[36.56,127.26]},
        {'loc':'LG CNS 부산', 'pos':[35.14,128.90]}, {'loc':'KT 목동', 'pos':[37.53,126.88]}
    ]
}

# 분석 범위(ROI): 각 그룹별 반경 20km 원형 결합
roi_new = ee.Geometry.MultiPolygon([ee.Geometry.Point([p[1], p[0]]).buffer(20000) for p in [x['pos'] for x in sites['New']]])
roi_old = ee.Geometry.MultiPolygon([ee.Geometry.Point([p[1], p[0]]).buffer(20000) for p in [x['pos'] for x in sites['Old']]])

# ==========================================
# 2. 🛰️ 위성 분석 (바다 투명화 적용)
# ==========================================
# 데이터 로드
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG").filterDate('2023-01-01','2023-12-31').select('avg_rad').mean()
lst = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterDate('2023-06-01','2023-09-30').select('ST_B10').mean().multiply(0.003418).add(149).subtract(273.15)
wc = ee.ImageCollection("ESA/WorldCover/v200").first().select('Map')
plants = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([129.35,37.24])), ee.Feature(ee.Geometry.Point([126.52,36.40])), ee.Feature(ee.Geometry.Point([129.29,35.32]))])

# GSI 모델링
norm = lambda img, mn, mx: img.clamp(mn, mx).subtract(mn).divide(mx - mn)
gsi = (ee.Image(1).subtract(norm(plants.distance(200000).divide(1000), 80, 150)).multiply(0.25)) \
      .add(ee.Image(1).subtract(norm(viirs, 0, 50)).multiply(0.20)) \
      .add(ee.Image(1).subtract(norm(lst, 20, 35)).multiply(0.15)) \
      .add(wc.remap([10,20,30,40,50,60,80], [0.3,0.7,1.0,0.6,0.0,1.0,0.0], 0.0).multiply(0.20)) \
      .add(ee.Image(1).subtract(norm(ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003")), 0, 15)).multiply(0.20))

# 🌊 바다 마스킹 (투명화)
mask = wc.neq(80)
tools = {
    '📊 GSI 적합도': (gsi.updateMask(mask), {'min':0.5, 'max':0.9, 'palette':['red','orange','yellow','green','blue']}),
    '💡 야간조도': (viirs.updateMask(mask), {'min':0, 'max':10, 'palette':['black','purple','cyan','white']}),
    '🌡️ 지표온도': (lst.updateMask(mask), {'min':20, 'max':35, 'palette':['blue','green','yellow','red']}),
    '🌲 토지피복': (wc.updateMask(mask), {'min':10, 'max':100, 'palette':['006400','ffbb22','fa0000','0064c8']}),
    '🏆 최적 후보지': (gsi.gt(0.78).updateMask(mask).selfMask(), {'palette':['00FF00']})
}

# ==========================================
# 3. 🗺️ 지도 및 기능 구현
# ==========================================
m = geemap.Map(center=[36.5, 127.5], zoom=7)

# [A] 모든 분석 레이어 미리 추가 (처음엔 모두 숨김 False)
# 레이어 이름 규칙: "{도구명}_{그룹명}" (예: "GSI 적합도_New_Data")
for t_name, (img, vis) in tools.items():
    m.addLayer(img.clip(roi_new), vis, f"{t_name}_New_Data", False)
    m.addLayer(img.clip(roi_old), vis, f"{t_name}_Old_Data", False)

# [B] 마커 그룹 추가
marker_groups = {'New': LayerGroup(name='M_New'), 'Old': LayerGroup(name='M_Old')}
for s in sites['New']: marker_groups['New'].add_layer(Marker(location=s['pos'], icon=AwesomeIcon(name='star', marker_color='green', icon_color='white'), title=s['loc'], draggable=False))
for s in sites['Old']: marker_groups['Old'].add_layer(Marker(location=s['pos'], icon=AwesomeIcon(name='building', marker_color='black', icon_color='white'), title=s['loc'], draggable=False))
m.add_layer(marker_groups['New']); m.add_layer(marker_groups['Old'])

# [C] 컨트롤러 (드롭다운 + 체크박스)
dropdown = widgets.Dropdown(options=tools.keys(), value='📊 GSI 적합도', description='🛠️ 도구:', layout=Layout(width='250px'))
chk_new = widgets.Checkbox(value=True, description='🟢 신규 후보지 (10곳)')
chk_old = widgets.Checkbox(value=True, description='⚫ 기존 데이터센터')
btn_osm = widgets.Button(description='⚡ 송전망 보기', icon='bolt', button_style='warning')

# [D] 핵심 작동 로직 (Update Function)
def update_layers(change=None):
    selected_tool = dropdown.value  # 현재 선택된 도구
    show_new = chk_new.value        # 신규 체크 여부
    show_old = chk_old.value        # 기존 체크 여부

    # 1. 마커 제어
    marker_groups['New'].visible = show_new
    marker_groups['Old'].visible = show_old

    # 2. 레이어 제어 (이름 매칭)
    for layer in m.layers:
        if not hasattr(layer, 'name'): continue

        # 신규 그룹 레이어 (_New_Data) -> "도구가 일치하고 & 신규 체크가 켜져야" 보임
        if layer.name.endswith('_New_Data'):
            layer.visible = (layer.name == f"{selected_tool}_New_Data") and show_new

        # 기존 그룹 레이어 (_Old_Data) -> "도구가 일치하고 & 기존 체크가 켜져야" 보임
        elif layer.name.endswith('_Old_Data'):
            layer.visible = (layer.name == f"{selected_tool}_Old_Data") and show_old

def load_osm(b):
    try:
        print("⏳ 송전선 로딩 중... (화면 중심 반경 20km)")
        lines = ox.features_from_point((m.center[0], m.center[1]), tags={'power':['line','cable']}, dist=20000)
        m.add_gdf(lines[lines.geometry.type.isin(['LineString','MultiLineString'])], layer_name='⚡ 송전선로', style={'color':'red', 'weight':3})
    except: print("⚠️ 송전선 없음")

# 이벤트 연결 및 초기 실행
dropdown.observe(update_layers, 'value')
chk_new.observe(update_layers, 'value')
chk_old.observe(update_layers, 'value')
btn_osm.on_click(load_osm)
update_layers() # 초기 상태 적용

print("✅ 준비 완료. 아래 체크박스를 끄면 해당 그룹의 [마커 + 원]이 동시에 사라집니다.")
display(VBox([dropdown, HBox([chk_new, chk_old]), btn_osm]))
display(m)

In [ ]:

# @title 🌏 대한민국 데이터센터 입지 분석 통합 (해남 + 강원) One Map
# [1] 필수 라이브러리 설치
!pip install -q geemap osmnx

import ee
import geemap
import osmnx as ox

# [2] 인증 및 초기화
try:
    ee.Initialize(project='YOUR_EARTH_ENGINE_PROJECT_ID')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='YOUR_EARTH_ENGINE_PROJECT_ID')

print("✅ 시스템 초기화 완료. 통합 분석(해남+강원)을 시작합니다.")

# ==========================================
# 🛠️ [공통 함수] 데이터 로딩 및 전처리
# ==========================================
def get_geo_data(roi):
    # 1. 전력수요 (VIIRS): 어두울수록 좋음
    viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
        .filterDate('2023-01-01', '2023-12-31').select('avg_rad').mean().clip(roi)

    # 2. 지표면 온도 (LST, Landsat 8): 여름철(6~9월)
    def cloud_mask(img):
        qa = img.select('QA_PIXEL')
        mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
        return img.updateMask(mask)

    lst = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(roi).filterDate('2023-06-01', '2023-09-30') \
        .map(cloud_mask).select('ST_B10').mean().clip(roi) \
        .multiply(0.00341802).add(149.0).subtract(273.15) # 섭씨 변환

    # 3. 토지피복 (ESA WorldCover)
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().clip(roi).select('Map')

    return viirs, lst, wc

# ==========================================
# 📍 [지역 1] 전라남도 해남군 (Opportunity Zone)
# ==========================================
print("🔹 [1/3] 전남 해남군 데이터 분석 중...")
roi_hn = ee.Geometry.Point([126.599, 34.574]).buffer(50000)
viirs_hn, lst_hn, wc_hn = get_geo_data(roi_hn)

# 해남 로직: 도심(Built-up, 50)이면서 + 전력 부하가 적은 곳(하위 80% 미만)
# 안전한 임계값 계산을 위해 서버 측 If문 사용
percentile_80 = viirs_hn.reduceRegion(ee.Reducer.percentile([80]), roi_hn, scale=500).get('avg_rad')
thresh = ee.Algorithms.If(percentile_80, percentile_80, 10)

low_power_mask = viirs_hn.lt(ee.Number(thresh))
built_up = wc_hn.eq(50)

# Opportunity Zone (바다 제외)
opp_zone = built_up.And(low_power_mask).updateMask(wc_hn.neq(80)).selfMask()

# ==========================================
# 📍 [지역 2] 강원도 영동 (GSI 모델링)
# ==========================================
print("🔹 [2/3] 강원 영동(강릉/동해/삼척) GSI 모델링 중...")
roi_gw = ee.Geometry.Point([129.00, 37.60]).buffer(40000)
viirs_gw, lst_gw, wc_gw = get_geo_data(roi_gw)

# GSI 5대 지표 계산
# ① 발전소 거리
plants = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([129.35, 37.24])), # 삼척
    ee.Feature(ee.Geometry.Point([129.14, 37.49])), # 동해
    ee.Feature(ee.Geometry.Point([128.99, 37.73]))  # 강릉
])
dist_score = ee.Image(1).subtract(plants.distance(200000).clip(roi_gw).divide(1000).subtract(80).divide(70)).clamp(0, 1)

# ②~⑤ 지표 (수요, 열, 토지, 경사)
demand_score = ee.Image(1).subtract(viirs_gw.clamp(0, 50).divide(50))
thermal_score = ee.Image(1).subtract(lst_gw.clamp(20, 35).subtract(20).divide(15))
land_score = wc_gw.remap([10, 20, 30, 40, 50, 60], [0.3, 0.7, 1.0, 0.6, 0.0, 1.0], 0.0)
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003").clip(roi_gw))
slope_score = ee.Image(0).where(slope.lt(15), 0.3).where(slope.lt(7), 0.7).where(slope.lt(3), 1.0)

# ⑥ GSI 가중합 및 바다 마스킹
gsi = (dist_score.multiply(0.25)).add(demand_score.multiply(0.20)).add(thermal_score.multiply(0.15)).add(land_score.multiply(0.20)).add(slope_score.multiply(0.20))
gsi_final = gsi.updateMask(wc_gw.neq(80)) # 바다(80) 제거
top_candidates = gsi_final.updateMask(gsi_final.gt(0.75)) # 상위 후보지

# ==========================================
# 🗺️ [3/3] 통합 지도 시각화 (One Map)
# ==========================================
print("🔹 [3/3] 통합 지도 생성 중...")

# 대한민국 중심 좌표로 지도 생성 (배경: OpenStreetMap.Mapnik)
m = geemap.Map(location=[36.5, 127.5], zoom_start=7)
m.add_basemap('OpenStreetMap')

# --- [전남 해남 레이어] ---
# 1. 야간 조도 (Power Load)
vis_night = {'min': 0, 'max': 10, 'palette': ['black', 'purple', 'cyan', 'white']}
m.addLayer(viirs_hn.updateMask(wc_hn.neq(80)), vis_night, '💡 [해남] Night Light (Power Load)', False)

# 2. 지표면 온도 (LST)
vis_lst = {'min': 20, 'max': 35, 'palette': ['blue', 'green', 'yellow', 'red']}
m.addLayer(lst_hn.updateMask(wc_hn.neq(80)), vis_lst, '🌡️ [해남] LST (Summer Temp)', False)

# 3. Opportunity Zone (최종 후보) - 빨간색
m.addLayer(opp_zone, {'palette': ['red']}, '📍 [해남] Opportunity Zone (Candidate)', True)

# --- [강원 영동 레이어] ---
# 4. 토지 피복 (Land Cover)
vis_wc = {'bands': ['Map'], 'min': 10, 'max': 100, 'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0']}
m.addLayer(wc_gw, vis_wc, '🌲 [강원] ESA Land Cover', False)

# 5. GSI 적합도 점수
vis_gsi = {'min': 0.4, 'max': 0.85, 'palette': ['red', 'orange', 'yellow', 'green', 'blue']}
m.addLayer(gsi_final, vis_gsi, '📊 [강원] GSI Score', True)

# 6. 최적 후보지 (Top Candidates)
m.addLayer(top_candidates, {'palette': ['00FF00']}, '🏆 [강원] Best Candidates', True)

# 7. 실제 송전망 (OSM)
try:
    print("   ⏳ [인프라] 강원도 송전망 데이터를 다운로드 중...")
    tags = {'power': ['line', 'tower', 'cable']}
    gdf = ox.features_from_point((37.75, 128.90), tags=tags, dist=25000)
    lines = gdf[gdf.geometry.type.isin(['LineString', 'MultiLineString'])]

    # geemap 방식(add_gdf)으로 벡터 추가 (빨간 실선)
    style = {'color': 'red', 'weight': 3}
    m.add_gdf(lines, layer_name='⚡ [강원] Power Lines (OSM)', style=style)
except Exception as e:
    print(f"⚠️ 송전선 데이터 로드 실패: {e}")

# 범례 및 레이어 컨트롤
m.add_legend(title="Land Cover", keys=['Forest', 'Cropland', 'Built-up', 'Water'], colors=['006400', 'f096ff', 'fa0000', '0064c8'])
m.addLayerControl()

print("\n🎉 통합 분석 완료! 지도 우측 상단 '레이어 아이콘'을 눌러 확인하세요.")
display(m)